# Faster R-CNN Helmet Detection Training
**Dataset:** Kaggle hard-hat-detection + Roboflow PPE (~6176 images)  
**Models trained:**
- `faster_rcnn_coco.pth` — COCO pretrained (ResNet50-FPN)
- `faster_rcnn_scratch.pth` — Trained from scratch (no pretrained weights)

Classes: `0=background`, `1=Helmet`, `2=No_Helmet`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, os, shutil, random, json
import xml.etree.ElementTree as ET
from pathlib import Path

subprocess.run(["pip", "install", "kaggle", "-q"])
os.makedirs("/root/.kaggle", exist_ok=True)

# ⚠️ Kaggle API bilgilerinizi buraya girin
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": "YOUR_KAGGLE_USERNAME", "key": "YOUR_KAGGLE_API_KEY"}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [ ]:
# Kaggle hard-hat-detection dataset'ini indir ve Roboflow ile birleştir
os.makedirs("/content/kaggle_raw", exist_ok=True)
subprocess.run(["kaggle", "datasets", "download", "-d", "andrewmvd/hard-hat-detection",
                "--unzip", "-p", "/content/kaggle_raw", "-q"])

for split in ["train", "valid", "test"]:
    os.makedirs("/content/merged/" + split + "/images", exist_ok=True)
    os.makedirs("/content/merged/" + split + "/labels", exist_ok=True)

CLASS_MAP = {"helmet": 0, "head": 1}

def get_yolo_lines(xml_path, W, H):
    lines = []
    for obj in ET.parse(xml_path).getroot().findall("object"):
        name = obj.find("name").text.lower().strip()
        if name not in CLASS_MAP:
            continue
        bb = obj.find("bndbox")
        xmin = float(bb.find("xmin").text)
        ymin = float(bb.find("ymin").text)
        xmax = float(bb.find("xmax").text)
        ymax = float(bb.find("ymax").text)
        cx = round((xmin + xmax) / 2 / W, 6)
        cy = round((ymin + ymax) / 2 / H, 6)
        bw = round((xmax - xmin) / W, 6)
        bh = round((ymax - ymin) / H, 6)
        lines.append(str(CLASS_MAP[name]) + " " + str(cx) + " " + str(cy) + " " + str(bw) + " " + str(bh))
    return lines

all_items = []

for xml_path in Path("/content/kaggle_raw/annotations").glob("*.xml"):
    img_path = "/content/kaggle_raw/images/" + xml_path.stem + ".png"
    if not os.path.exists(img_path):
        continue
    root = ET.parse(str(xml_path)).getroot()
    size = root.find("size")
    if size is None:
        continue
    W = int(size.find("width").text)
    H = int(size.find("height").text)
    if W == 0 or H == 0:
        continue
    lines = get_yolo_lines(str(xml_path), W, H)
    if lines:
        all_items.append(("kgl", img_path, lines))

# ⚠️ Drive'daki Roboflow PPE dataset yolunu kendinize göre güncelleyin
OLD = "/content/drive/Othercomputers/MacBook Pro'm/Desktop/ppe detection.yolov8/train"

for img in Path(OLD + "/images").glob("*.*"):
    if img.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue
    lbl = Path(OLD + "/labels") / (img.stem + ".txt")
    if not lbl.exists():
        continue
    with open(str(lbl)) as f:
        content = f.read().strip()
    if content:
        all_items.append(("old", str(img), content.split("\n")))

print("Toplam: " + str(len(all_items)))

random.seed(42)
random.shuffle(all_items)
n = len(all_items)
boundaries = [0, int(n * 0.7), int(n * 0.9), n]
split_names = ["train", "valid", "test"]

for i, split in enumerate(split_names):
    chunk = all_items[boundaries[i]:boundaries[i+1]]
    for item in chunk:
        src_type, img_path, lines = item
        fname = Path(img_path).stem
        ext = Path(img_path).suffix
        shutil.copy(img_path, "/content/merged/" + split + "/images/" + fname + ext)
        lbl_path = "/content/merged/" + split + "/labels/" + fname + ".txt"
        with open(lbl_path, "w") as f:
            f.write("\n".join(lines))
    print(split + ": " + str(len(chunk)))

shutil.rmtree("/content/kaggle_raw")
print("Veri hazir!")

In [ ]:
import torch
import torchvision
import cv2
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

class PPEYoloDataset(Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = sorted([f for f in os.listdir(image_dir)
                                   if f.endswith(('.jpg', '.jpeg', '.png'))])

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img = cv2.imread(os.path.join(self.image_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        height, width, _ = img.shape

        label_path = os.path.join(self.label_dir, os.path.splitext(img_name)[0] + '.txt')
        boxes, labels = [], []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.split()
                    if len(parts) == 5:
                        class_id = int(parts[0]) + 1  # 0=background, 1=helmet, 2=no_helmet
                        x_c, y_c, w, h = map(float, parts[1:])
                        xmin = (x_c - w / 2) * width
                        ymin = (y_c - h / 2) * height
                        xmax = (x_c + w / 2) * width
                        ymax = (y_c + h / 2) * height
                        if xmax > xmin and ymax > ymin:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(class_id)

        if len(boxes) == 0:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64),
        }
        return torchvision.transforms.functional.to_tensor(img), target

    def __len__(self):
        return len(self.image_files)

def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = PPEYoloDataset("/content/merged/train/images", "/content/merged/train/labels")
valid_dataset = PPEYoloDataset("/content/merged/valid/images", "/content/merged/valid/labels")
test_dataset  = PPEYoloDataset("/content/merged/test/images",  "/content/merged/test/labels")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Device: {device}")
print(f"Train: {len(train_dataset)}  Valid: {len(valid_dataset)}  Test: {len(test_dataset)}")

## Model 1: COCO Pretrained

In [ ]:
def build_model(pretrained=True):
    weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.COCO_V1 if pretrained else None
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=3)
    return model

def train_model(model, train_loader, num_epochs=10, lr=0.005, label="model"):
    model.to(device)
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, momentum=0.9, weight_decay=0.0005
    )
    print(f"\n🚀 {label} eğitimi başlıyor...")
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for images, targets in train_loader:
            images  = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss_dict.values())
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            epoch_loss += losses.item()
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss / len(train_loader):.4f}")
    print(f"✅ {label} eğitimi tamamlandı!")
    return model

model_coco = build_model(pretrained=True)
model_coco = train_model(model_coco, train_loader, num_epochs=10, label="COCO Pretrained")

torch.save(model_coco.state_dict(), "faster_rcnn_coco.pth")
print("Kaydedildi: faster_rcnn_coco.pth")

## Model 2: From Scratch (No Pretrained Weights)

In [ ]:
model_scratch = build_model(pretrained=False)
model_scratch = train_model(model_scratch, train_loader, num_epochs=10, lr=0.005, label="From Scratch")

torch.save(model_scratch.state_dict(), "faster_rcnn_scratch.pth")
print("Kaydedildi: faster_rcnn_scratch.pth")

## Evaluation (mAP50 + FPS)

In [ ]:
!pip install -q torchmetrics
import time
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def evaluate(model, loader, label="Model"):
    model.eval()
    metric = MeanAveragePrecision(iou_thresholds=[0.5])
    total_time = 0
    total_images = 0

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            t0 = time.time()
            outputs = model(images)
            total_time += time.time() - t0
            total_images += len(images)

            preds = [{"boxes": o["boxes"].cpu(), "scores": o["scores"].cpu(), "labels": o["labels"].cpu()} for o in outputs]
            gts   = [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]
            metric.update(preds, gts)

    result = metric.compute()
    fps = total_images / total_time

    print(f"\n{'='*50}")
    print(f"📊 {label} TEST SONUÇLARI")
    print(f"{'='*50}")
    print(f"Test Edilen Görsel : {total_images}")
    print(f"mAP@50             : {result['map_50'].item()*100:.2f}%")
    print(f"FPS                : {fps:.2f}")
    print(f"{'='*50}")

evaluate(model_coco,    test_loader, label="Faster R-CNN (COCO Pretrained)")
evaluate(model_scratch, test_loader, label="Faster R-CNN (From Scratch)")

In [ ]:
# ⚠️ Drive'a kaydetmek için yolu kendinize göre güncelleyin
import shutil
shutil.copy("faster_rcnn_coco.pth",    "/content/drive/MyDrive/faster_rcnn_coco.pth")
shutil.copy("faster_rcnn_scratch.pth", "/content/drive/MyDrive/faster_rcnn_scratch.pth")
print("Modeller Drive'a kaydedildi!")